# Edge–Cloud Collaborative Scheduling Lab

This notebook is a runnable companion to the
[Codeforces 2251A problem](https://codeforces.com/contest/2251/problem/A).
It keeps four activities in one place:

1. understand the system and interactive protocol;
2. connect those concepts to both the frozen baseline and the current layered scheduler;
3. run the policy against deterministic scenarios; and
4. add optimizations one at a time and measure what actually improves.

The repository remains the source of truth. The notebook reads the checked-in C++ source,
scenarios, task-time table, local judge, and baseline benchmark instead of copying them into
a disconnected toy implementation.

## Goal

By the end of this lab, we should be able to answer:

- What work runs on the edge, in a cloud, and on the shared links?
- What does one request do from `ARR` to `FIN`?
- What does an assignment such as `E D PRE -1 3 7 12 19` mean?
- Why is the frozen baseline correct but intentionally inefficient?
- When does decode grouping help, and when can waiting for a group hurt?
- Which scenario should expose each optimization?
- Did a code change remain legal, and did it improve score, throughput, TDR, or TPOT?

**Notebook mode:** tutorial + experiment log.  
**Reader:** someone learning the problem while implementing a contest scheduler.  
**Handoff:** a top-to-bottom executable notebook tied to the current repository.

## Setup

In [1]:
from __future__ import annotations

import html
import json
import math
import re
import subprocess
from pathlib import Path
from typing import Any, Iterable

from IPython.display import Code, HTML, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts in the root or notebooks/ directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "main.cpp").is_file() and (candidate / "tools/local_judge.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find main.cpp and tools/local_judge.py above the working directory")


REPO_ROOT = find_repo_root()
BUILD_DIR = REPO_ROOT / "build"
BASELINE_SOLVER = BUILD_DIR / "v0-baseline"
WORKING_SOLVER = BUILD_DIR / "scheduler"
SCENARIO_DIR = REPO_ROOT / "scenarios"
BASELINE_SNAPSHOT = REPO_ROOT / "benchmarks/baseline-v0.json"
REGISTRY_PATH = REPO_ROOT / "scheduler_versions/registry.json"

print(f"Repository: {REPO_ROOT}")
print(f"Frozen v0:  {BASELINE_SOLVER}")
print(f"Current v7: {WORKING_SOLVER}")

Repository: /Users/aadikhanna/Github/AI Optimizer/CodeForces-Edge-Collaborative-Scheduling
Frozen v0:  /Users/aadikhanna/Github/AI Optimizer/CodeForces-Edge-Collaborative-Scheduling/build/v0-baseline
Current v7: /Users/aadikhanna/Github/AI Optimizer/CodeForces-Edge-Collaborative-Scheduling/build/scheduler


In [2]:
def run_checked(command: list[str], timeout_seconds: float = 120.0) -> subprocess.CompletedProcess[str]:
    """Run a bounded command in the repository and show concise output."""
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        timeout=timeout_seconds,
    )
    if completed.stdout.strip():
        print(completed.stdout.rstrip())
    if completed.returncode != 0:
        if completed.stderr.strip():
            print(completed.stderr.rstrip())
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {' '.join(command)}")
    return completed


run_checked(["make", "build/v0-baseline", "build/scheduler"])
assert BASELINE_SOLVER.is_file(), "The frozen baseline executable was not created"
assert WORKING_SOLVER.is_file(), "The current scheduler executable was not created"

make[1]: `build/v0-baseline' is up to date.
make[1]: `build/scheduler' is up to date.


In [3]:
def display_table(rows: Iterable[dict[str, Any]], columns: list[tuple[str, str]] | None = None) -> None:
    """Render a small list of dictionaries without requiring pandas."""
    bounded_rows = list(rows)
    if not bounded_rows:
        display(Markdown("_No rows._"))
        return
    if columns is None:
        columns = [(key, key) for key in bounded_rows[0]]
    header = "".join(f"<th>{html.escape(label)}</th>" for _, label in columns)
    body = []
    for row in bounded_rows:
        cells = "".join(
            f"<td>{html.escape(str(row.get(key, '')))}</td>" for key, _ in columns
        )
        body.append(f"<tr>{cells}</tr>")
    display(
        HTML(
            "<table><thead><tr>"
            + header
            + "</tr></thead><tbody>"
            + "".join(body)
            + "</tbody></table>"
        )
    )

## 1. Build the mental model

Each request has a **prefill phase** followed by one or more **decode iterations**.

```text
ARR
  │
  ▼
Edge:  P PRE ──UP──▶ Cloud: P PROC ──DOWN──▶ Edge: P POST
                                                   │
                                                   ▼
Edge:  D PRE ──UP──▶ Cloud: D PROC ──DOWN──▶ Edge: D POST
         ▲                                           │
         └──────── next token if not FIN ────────────┘
```

Resource constraints:

- There is one edge compute server, `E`.
- There are `K` cloud compute servers, `C0 ... C(K-1)`.
- Every server can run at most one task at a time.
- All clouds collectively share one FIFO `UP` transfer queue and one FIFO `DOWN` queue.
- Transfers do not occupy edge or cloud compute, but competing transfers queue on their link.
- A request is assigned to a cloud by its `P PRE` task and keeps that cloud association.

### Prefill versus decode grouping

| Family | Edge stage | Cloud stage | Groupable? |
|---|---|---|---|
| Prefill (`P`) | `P PRE`, `P POST` | `P PROC` | No: each assignment names one request |
| Decode (`D`) | `D PRE`, `D POST` | `D PROC` | Yes: assignments carry a list of request IDs |

Grouping is therefore **not cloud-only**. Decode preprocessing and postprocessing can be
grouped on the edge, while decode processing can be grouped on a cloud. A `D PROC` group
must contain requests assigned to that same cloud.

A group is also **not permanent**. It represents one stage of one decode iteration. After
`D POST`, unfinished requests become eligible for their next token and may be regrouped.
A request that reaches its hidden output length emits `FIN` and is absent from future groups.

### Reading one assignment

```text
E D PRE -1 3 7 12 19
│ │  │   │ │ └────── request IDs in this group
│ │  │   │ └──────── group size = 3
│ │  │   └────────── required placeholder for edge decode work
│ │  └────────────── preprocessing stage
│ └───────────────── decode family
└─────────────────── run on the edge server
```

This starts one grouped decode-preprocessing task for requests `7`, `12`, and `19`.
It does not mean “run from time 7 to time 19,” and `-1` is not a cloud ID.

## 2. Inspect the workload suite

Scenario JSON includes system parameters, scoring weights, a task-time table, and requests.
`output_length` is judge-only truth: the local judge uses it to decide when to emit `FIN`,
but the scheduler only sees `ARR request_id input_length`.

In [4]:
scenario_paths = sorted(SCENARIO_DIR.glob("*.json"))
scenarios = {path.stem: json.loads(path.read_text()) for path in scenario_paths}

pressure_by_name = {
    "official_worked_example": "calibration",
    "single_sanity": "one-request lifecycle",
    "two_cloud_parallel": "parallelism / reservations",
    "output_length_skew": "hidden output skew",
    "batch_friendly_burst": "decode grouping",
    "latency_sensitive_stream": "throughput vs latency",
    "link_bottleneck": "shared UP/DOWN queues",
    "prefill_preemption": "prefill layer chunking",
    "degenerate_one_layer": "minimum legal mechanics",
    "interpolation_missing_values": "task-time interpolation",
    "single_cloud_prefill_interleave": "adaptive prefill chunking",
    "slo_priority_collision": "SLO-aware urgency",
    "latency_weighted_slow_link": "latency-weighted link policy",
    "nonmonotonic_batch_table": "table-aware group size",
}

scenario_summary = []
for path in scenario_paths:
    data = scenarios[path.stem]
    requests = data["requests"]
    scenario_summary.append(
        {
            "scenario": data["name"],
            "K": data["system"]["K"],
            "S": data["system"]["S"],
            "requests": len(requests),
            "tokens": sum(request["output_length"] for request in requests),
            "w_tp": data["scoring"]["w_tp"],
            "w_c": data["scoring"]["w_c"],
            "pressure": pressure_by_name.get(data["name"], "general policy behavior"),
        }
    )

display_table(
    scenario_summary,
    [
        ("scenario", "Scenario"),
        ("K", "Clouds"),
        ("S", "Schedule cost"),
        ("requests", "Requests"),
        ("tokens", "Hidden output tokens"),
        ("w_tp", "Throughput weight"),
        ("w_c", "Latency weight"),
        ("pressure", "Designed to expose"),
    ],
)

Scenario,Clouds,Schedule cost,Requests,Hidden output tokens,Throughput weight,Latency weight,Designed to expose
official_worked_example,1,1.0,1,1,0.5,0.5,calibration
single_sanity,1,1.0,1,3,0.5,0.5,one-request lifecycle
two_cloud_parallel,2,1.0,6,29,0.5,0.5,parallelism / reservations
output_length_skew,2,1.5,6,56,0.6,0.4,hidden output skew
batch_friendly_burst,4,8.0,16,128,0.9,0.1,decode grouping
latency_sensitive_stream,3,2.0,12,60,0.2,0.8,throughput vs latency
link_bottleneck,4,1.0,8,32,0.6,0.4,shared UP/DOWN queues
prefill_preemption,2,1.0,5,64,0.5,0.5,prefill layer chunking
degenerate_one_layer,1,1.0,3,6,0.5,0.5,minimum legal mechanics
interpolation_missing_values,2,1.0,4,10,0.5,0.5,task-time interpolation


### Choose a scenario to study

Change `SELECTED_SCENARIO_FILE`, rerun this cell, and then rerun the experiment cells below.

In [5]:
SELECTED_SCENARIO_FILE = "04_batch_friendly_burst.json"

selected_path = SCENARIO_DIR / SELECTED_SCENARIO_FILE
selected = json.loads(selected_path.read_text())

display(Markdown(f"### `{selected['name']}`\n\n{selected['description']}"))
display(Code(json.dumps({"system": selected["system"], "scoring": selected["scoring"]}, indent=2), language="json"))

request_preview = []
for request_id, request in enumerate(selected["requests"][:12]):
    request_preview.append(
        {
            "request_id": request_id,
            "arrival": request["arrival"],
            "input_length (visible)": request["input_length"],
            "output_length (hidden)": request["output_length"],
        }
    )
display_table(request_preview)
if len(selected["requests"]) > len(request_preview):
    print(f"Showing {len(request_preview)} of {len(selected['requests'])} requests.")

### `batch_friendly_burst`

A high-overhead request burst where decode grouping should materially improve throughput.

{
  "system": {
    "K": 4,
    "S": 8.0,
    "latency_in_ms": 0.2,
    "bandwidth_gbps": 100.0,
    "bytes_per_token": 512,
    "num_layers": 8
  },
  "scoring": {
    "SLO1": 600.0,
    "SLO2": 100.0,
    "tp_UB": 0.35,
    "tp_base": 0.01,
    "dist_base": 5.0,
    "w_tp": 0.9,
    "w_c": 0.1
  }
}

request_id,arrival,input_length (visible),output_length (hidden)
0,0.0,16,8
1,0.0,16,8
2,0.0,16,8
3,0.0,16,8
4,0.0,16,8
5,0.0,16,8
6,0.0,16,8
7,0.0,16,8
8,0.0,16,8
9,0.0,16,8


Showing 12 of 16 requests.


## 3. Connect the model to the code

We keep two distinct artifacts:

- `scheduler_versions/v0_baseline.cpp` is the frozen, deliberately simple reference;
- `main.cpp` is the current layer-20 engine with promoted v25/v27/v33/v41/v43/v53 revisions and is identical
  to `scheduler_versions/layered_scheduler.cpp` with its default `OPT_LEVEL=20`.

The frozen baseline is a state machine plus three FIFO structures:

- `pending_requests_`: arrived requests that do not yet have a cloud reservation;
- `edge_ready_`: legal edge tasks ordered by when they became ready; and
- `cloud_ready_[cloud]`: legal cloud tasks for each cloud.

It adds one deliberate restriction: it reserves an entire cloud for a request
from `P PRE` until `FIN`. The contest does not require that restriction. It makes the first
implementation easy to reason about, but it leaves clouds idle while their reserved request
is on the edge or waiting for a transfer.

In [6]:
baseline_source = (REPO_ROOT / "scheduler_versions/v0_baseline.cpp").read_text()
layered_source = (REPO_ROOT / "scheduler_versions/layered_scheduler.cpp").read_text()


def source_between(
    source: str, start_marker: str, end_marker: str, max_lines: int = 180
) -> str:
    start = source.index(start_marker)
    end = source.index(end_marker, start)
    snippet = source[start:end].rstrip()
    lines = snippet.splitlines()
    if len(lines) > max_lines:
        lines = lines[:max_lines] + ["// ... bounded notebook preview ..."]
    return "\n".join(lines)


display(Markdown("### Request states"))
display(
    Code(
        source_between(baseline_source, "enum class RequestState", "enum class TaskKind"),
        language="cpp",
    )
)

### Request states

enum class RequestState {
    UNSEEN,
    WAITING_FOR_CLOUD,
    P_PRE_RUNNING,
    WAITING_PREFILL_UP,
    P_PROC_READY,
    P_PROC_RUNNING,
    WAITING_PREFILL_DOWN,
    P_POST_READY,
    P_POST_RUNNING,
    D_PRE_READY,
    D_PRE_RUNNING,
    WAITING_DECODE_UP,
    D_PROC_READY,
    D_PROC_RUNNING,
    WAITING_DECODE_DOWN,
    D_POST_READY,
    D_POST_RUNNING,
    FINISHED,
};

In [7]:
display(Markdown("### Baseline admission and edge dispatch"))
display(
    Code(
        source_between(
            baseline_source, "string dispatch_admission()", "string dispatch_cloud_task"
        ),
        language="cpp",
    )
)

### Baseline admission and edge dispatch

string dispatch_admission() {
        int request_id = pending_requests_.front();
        pending_requests_.pop_front();

        int cloud = free_clouds_.front();
        free_clouds_.pop_front();

        Request& req = request(request_id);
        expect_state(req, RequestState::WAITING_FOR_CLOUD, "P PRE dispatch");
        if (cloud_reserved_[cloud]) {
            fail("admission selected a reserved cloud");
        }

        req.cloud = cloud;
        req.state = RequestState::P_PRE_RUNNING;
        cloud_reserved_[cloud] = true;
        edge_busy_ = true;

        return "E P PRE " + to_string(cloud) + " " + to_string(request_id);
    }

    string dispatch_edge_task() {
        ReadyTask task = edge_ready_.front();
        edge_ready_.pop_front();
        Request& req = request(task.request_id);

        edge_busy_ = true;
        switch (task.kind) {
            case TaskKind::P_POST:
                expect_state(req, RequestState::P_POST_READY, "P POST dispatch");
                req.state = RequestState::P_POST_RUNNING;
                return "E P POST " + to_string(req.cloud) + " " + to_string(req.id);
            case TaskKind::D_PRE:
                expect_state(req, RequestState::D_PRE_READY, "D PRE dispatch");
                req.state = RequestState::D_PRE_RUNNING;
                return "E D PRE -1 1 " + to_string(req.id);
            case TaskKind::D_POST:
                expect_state(req, RequestState::D_POST_READY, "D POST dispatch");
                req.state = RequestState::D_POST_RUNNING;
                return "E D POST -1 1 " + to_string(req.id);
            case TaskKind::P_PROC:
            case TaskKind::D_PROC:
                fail("cloud task appeared in the edge queue");
        }
        fail("unreachable edge task kind");
    }

In [8]:
display(Markdown("### The central dispatch decision"))
display(
    Code(
        source_between(
            baseline_source, "vector<string> dispatch_ready_work()", "void print_response"
        ),
        language="cpp",
    )
)

### The central dispatch decision

vector<string> dispatch_ready_work() {
        vector<string> assignments;
        assignments.reserve(cloud_count_ + 1);

        if (!edge_busy_) {
            const bool admission_available =
                !pending_requests_.empty() && !free_clouds_.empty();
            const bool edge_task_available = !edge_ready_.empty();

            if (admission_available || edge_task_available) {
                bool choose_admission = false;
                if (!edge_task_available) {
                    choose_admission = true;
                } else if (admission_available) {
                    const Request& pending = request(pending_requests_.front());
                    choose_admission =
                        pending.admission_sequence <= edge_ready_.front().sequence;
                }

                if (choose_admission) {
                    assignments.push_back(dispatch_admission());
                } else {
                    assignments.push_back(dispatch_edge_task());
                }
            }
        }

        for (int cloud = 0; cloud < cloud_count_; ++cloud) {
            if (!cloud_busy_[cloud] && !cloud_ready_[cloud].empty()) {
                assignments.push_back(dispatch_cloud_task(cloud));
            }
        }

        if (assignments.size() > static_cast<size_t>(cloud_count_ + 1)) {
            fail("attempted too many assignments in one response");
        }
        return assignments;
    }

### What makes this a baseline?

The frozen v0 code intentionally does **none** of the following:

- multiple active requests on one cloud;
- load-aware cloud selection;
- grouped decode tasks;
- task-time-table-based batch selection;
- SLO-aware task priority;
- controlled waiting to form a better group;
- layer-chunked prefill; or
- indirect link-aware scheduling.

That is useful experimentally: every later layer has one primary mechanism and a scenario
designed to make that mechanism visible. The layered engine uses compile-time feature gates,
so `OPT_LEVEL=4` contains layers 1 through 4 but none of the later gates. The rejected learned
layers remain reproducible, while the current submission follows the promoted v15 → v19 → v20
terminal-stage branch.

## 4. Understand the task-time table

For prefill columns, the lookup size is the request's input length. For decode columns, it
is the decode group size. Missing values (`-1`) are ignored and intermediate sizes are
linearly interpolated by the local judge.

In [9]:
def task_rows_for(scenario_path: Path, scenario: dict[str, Any]) -> list[dict[str, float]]:
    if "task_times" in scenario:
        return scenario["task_times"]
    profile_path = scenario_path.parent / scenario["task_times_file"]
    return json.loads(profile_path.read_text())["task_times"]


task_rows = task_rows_for(selected_path, selected)
display_table(task_rows[:8])

batch_size,prefill_pre,prefill_proc,prefill_post,decode_pre,decode_proc,decode_post
1,0.3,2.0,0.3,0.5,3.0,0.4
4,0.35,3.0,0.32,0.9,7.0,0.7
8,0.4,4.5,0.35,1.3,11.0,1.0
16,0.5,7.0,0.4,2.0,18.0,1.5
32,0.7,12.0,0.5,3.2,30.0,2.4
64,1.0,22.0,0.7,5.5,52.0,4.0
256,2.5,80.0,1.5,18.0,180.0,14.0
4096,20.0,1200.0,10.0,200.0,2000.0,150.0


In [10]:
TASK_COLUMNS = (
    "prefill_pre",
    "prefill_proc",
    "prefill_post",
    "decode_pre",
    "decode_proc",
    "decode_post",
)


def interpolate_duration(rows: list[dict[str, float]], column: str, size: int) -> float:
    points = sorted(
        (int(row["batch_size"]), float(row[column]))
        for row in rows
        if float(row[column]) >= 0
    )
    if not points:
        raise ValueError(f"No usable values for {column}")
    if size <= points[0][0]:
        return points[0][1]
    if size >= points[-1][0]:
        return points[-1][1]
    for (left_size, left_value), (right_size, right_value) in zip(points, points[1:]):
        if size == left_size:
            return left_value
        if left_size < size < right_size:
            fraction = (size - left_size) / (right_size - left_size)
            return left_value + fraction * (right_value - left_value)
    raise AssertionError("Interpolation should have returned")


interpolation_demo = [
    {
        "size": size,
        **{column: round(interpolate_duration(task_rows, column, size), 4) for column in TASK_COLUMNS},
    }
    for size in (1, 2, 4, 6, 8, 16)
]
display_table(interpolation_demo)

size,prefill_pre,prefill_proc,prefill_post,decode_pre,decode_proc,decode_post
1,0.3,2.0,0.3,0.5,3.0,0.4
2,0.3167,2.3333,0.3067,0.6333,4.3333,0.5
4,0.35,3.0,0.32,0.9,7.0,0.7
6,0.375,3.75,0.335,1.1,9.0,0.85
8,0.4,4.5,0.35,1.3,11.0,1.0
16,0.5,7.0,0.4,2.0,18.0,1.5


### A first grouping estimate

The following is a **local service-cost estimate**, not a complete scheduler simulation.
For a decode group of size `b`, it adds:

1. the scheduling cost `S` for each of `D PRE`, `D PROC`, and `D POST`;
2. the interpolated task time for those three stages; and
3. one upload and one download for `b` decode items.

It deliberately ignores queueing, overlap with other resources, and time spent waiting for
requests to become group-compatible. Those effects are why we still need the dynamic judge.

In [11]:
def transfer_time_ms(system: dict[str, Any], item_count: int) -> float:
    size_bytes = item_count * int(system["bytes_per_token"])
    return float(system["latency_in_ms"]) + 8.0 * size_bytes / (
        float(system["bandwidth_gbps"]) * 1_000_000.0
    )


def estimated_decode_cycle_ms(
    system: dict[str, Any], rows: list[dict[str, float]], group_size: int
) -> float:
    compute = sum(
        float(system["S"]) + interpolate_duration(rows, column, group_size)
        for column in ("decode_pre", "decode_proc", "decode_post")
    )
    transfers = 2.0 * transfer_time_ms(system, group_size)
    return compute + transfers


candidate_sizes = [size for size in (1, 2, 4, 8, 16) if size <= len(selected["requests"])]
singleton_cycle = estimated_decode_cycle_ms(selected["system"], task_rows, 1)
group_estimates = []
for size in candidate_sizes:
    group_cycle = estimated_decode_cycle_ms(selected["system"], task_rows, size)
    group_estimates.append(
        {
            "group_size": size,
            "estimated cycle ms": round(group_cycle, 4),
            "ms per request": round(group_cycle / size, 4),
            "idealized throughput gain": f"{size * singleton_cycle / group_cycle:.2f}x",
        }
    )
display_table(group_estimates)

group_size,estimated cycle ms,ms per request,idealized throughput gain
1,28.3001,28.3001,1.00x
2,29.8668,14.9334,1.90x
4,33.0003,8.2501,3.43x
8,37.7007,4.7126,6.01x
16,45.9013,2.8688,9.86x


The largest currently ready group often minimizes service time per request, but that does
**not** prove we should always wait for the largest possible group:

- compatible requests may not be ready yet;
- `D PROC` members must belong to the same cloud;
- waiting increases request age and can violate TDR/TPOT targets;
- a larger group consumes a resource for longer and may block urgent work;
- shared-link queueing can dominate the isolated estimate; and
- the task-time table itself may show weak or negative scaling at larger sizes.

The correct loop is therefore: use the table to form a hypothesis, then use the dynamic
judge to measure the complete policy.

## 5. Run the baseline on one case

The local judge sends startup data and event frames to the actual C++ executable. It accepts
any legal policy decision and generates the resulting `TDN`, `XDN`, and `FIN` events. This is
different from replaying one fixed transcript.

In [12]:
selected_result_path = BUILD_DIR / "notebook-selected-result.json"
run_checked(
    [
        "python3",
        "tools/local_judge.py",
        "--solver",
        str(BASELINE_SOLVER),
        "--scenarios",
        str(selected_path),
        "--json-out",
        str(selected_result_path),
    ]
)
selected_result = json.loads(selected_result_path.read_text())[0]
display_table([selected_result])

PASS batch_friendly_burst         score= 200.433 tp=0.052597 tdr=969.725 tpot=68.663 elapsed=2433.600


scenario,description,legal,score,throughput,tdr,tpot,distance,elapsed,tokens,frames,scheduler_cpu_seconds,judge_wall_seconds
batch_friendly_burst,A high-overhead request burst where decode grouping should materially improve throughput.,True,200.43253364659435,0.05259697567389861,969.7250000000009,68.6625000000002,0.6162083333333349,2433.6000000000067,128,721,0.010275999999999999,0.025571500009391457


### Reconstruct the score

The score combines normalized throughput with an SLO-compliance component. Higher score and
throughput are better; lower TDR, TPOT, distance, and elapsed time are better.

In [13]:
def clamp01(value: float) -> float:
    return max(0.0, min(1.0, value))


def reconstruct_score(result: dict[str, Any], scoring: dict[str, Any]) -> dict[str, float]:
    excess_tdr = max(0.0, (result["tdr"] - scoring["SLO1"]) / scoring["SLO1"])
    excess_tpot = max(0.0, (result["tpot"] - scoring["SLO2"]) / scoring["SLO2"])
    distance = math.hypot(excess_tdr, excess_tpot)
    throughput_component = clamp01(
        (result["throughput"] - scoring["tp_base"])
        / (scoring["tp_UB"] - scoring["tp_base"])
    )
    distance_base = scoring["dist_base"]
    waiting_component = (
        max(0.0, 1.0 - distance / distance_base)
        if distance_base > 0
        else (1.0 if distance == 0 else 0.0)
    )
    score = 1000.0 * (
        scoring["w_tp"] * throughput_component + scoring["w_c"] * waiting_component
    )
    return {
        "throughput component": throughput_component,
        "SLO distance": distance,
        "SLO component": waiting_component,
        "reconstructed score": score,
    }


score_parts = reconstruct_score(selected_result, selected["scoring"])
display_table([{key: round(value, 6) for key, value in score_parts.items()}])
assert math.isclose(
    score_parts["reconstructed score"], selected_result["score"], rel_tol=0, abs_tol=1e-7
)

throughput component,SLO distance,SLO component,reconstructed score
0.125285,0.616208,0.876758,200.432534


## 6. Run the complete scenario suite

In [14]:
suite_result_path = BUILD_DIR / "notebook-baseline-results.json"
run_checked(
    [
        "python3",
        "tools/local_judge.py",
        "--solver",
        str(BASELINE_SOLVER),
        "--scenarios",
        str(SCENARIO_DIR),
        "--json-out",
        str(suite_result_path),
    ]
)
suite_results = json.loads(suite_result_path.read_text())

display_table(
    [
        {
            "scenario": row["scenario"],
            "score": f"{row['score']:.3f}",
            "throughput": f"{row['throughput']:.6f}",
            "TDR": f"{row['tdr']:.3f}",
            "TPOT": f"{row['tpot']:.3f}",
            "elapsed": f"{row['elapsed']:.3f}",
        }
        for row in suite_results
    ]
)

PASS official_worked_example      score= 500.000 tp=0.022222 tdr=30.000 tpot=0.000 elapsed=45.000
PASS single_sanity                score= 754.111 tp=0.081151 tdr=10.263 tpot=8.902 elapsed=36.968
PASS two_cloud_parallel           score= 776.939 tp=0.127224 tdr=90.035 tpot=10.950 elapsed=227.944
PASS output_length_skew           score= 724.137 tp=0.128046 tdr=96.372 tpot=10.538 elapsed=437.344
PASS batch_friendly_burst         score= 200.433 tp=0.052597 tdr=969.725 tpot=68.663 elapsed=2433.600
PASS latency_sensitive_stream     score= 591.985 tp=0.158490 tdr=106.847 tpot=14.694 elapsed=378.572
PASS link_bottleneck              score= 243.119 tp=0.001185 tdr=14669.700 tpot=227.167 elapsed=27013.300
PASS prefill_preemption           score= 704.428 tp=0.097651 tdr=362.957 tpot=8.907 elapsed=655.396
PASS degenerate_one_layer         score= 766.371 tp=0.084584 tdr=25.970 tpot=7.901 elapsed=70.936
PASS interpolation_missing_values score= 761.047 tp=0.109198 tdr=42.289 tpot=8.909 elapsed=91.577

scenario,score,throughput,TDR,TPOT,elapsed
official_worked_example,500.000,0.022222,30.000,0.000,45.000
single_sanity,754.111,0.081151,10.263,8.902,36.968
two_cloud_parallel,776.939,0.127224,90.035,10.950,227.944
output_length_skew,724.137,0.128046,96.372,10.538,437.344
batch_friendly_burst,200.433,0.052597,969.725,68.663,2433.600
latency_sensitive_stream,591.985,0.158490,106.847,14.694,378.572
link_bottleneck,243.119,0.001185,14669.700,227.167,27013.300
prefill_preemption,704.428,0.097651,362.957,8.907,655.396
degenerate_one_layer,766.371,0.084584,25.970,7.901,70.936
interpolation_missing_values,761.047,0.109198,42.289,8.909,91.577


### Check reproducibility against the saved baseline

In [15]:
saved_baseline = {row["scenario"]: row for row in json.loads(BASELINE_SNAPSHOT.read_text())}
current_baseline = {row["scenario"]: row for row in suite_results}

comparison_rows = []
maximum_score_delta = 0.0
for scenario_name, expected in saved_baseline.items():
    actual = current_baseline[scenario_name]
    score_delta = actual["score"] - expected["score"]
    maximum_score_delta = max(maximum_score_delta, abs(score_delta))
    comparison_rows.append(
        {
            "scenario": scenario_name,
            "legal": actual["legal"],
            "score delta": f"{score_delta:+.9f}",
            "throughput delta": f"{actual['throughput'] - expected['throughput']:+.9f}",
        }
    )

display_table(comparison_rows)
assert all(row["legal"] for row in suite_results)
assert maximum_score_delta < 1e-6
print("Reproducibility check passed: frozen v0 results match baseline-v0.json.")

scenario,legal,score delta,throughput delta
official_worked_example,True,+0.000000000,+0.000000000
single_sanity,True,+0.000000000,+0.000000000
two_cloud_parallel,True,+0.000000000,+0.000000000
output_length_skew,True,+0.000000000,+0.000000000
batch_friendly_burst,True,+0.000000000,+0.000000000
latency_sensitive_stream,True,+0.000000000,+0.000000000
link_bottleneck,True,+0.000000000,+0.000000000
prefill_preemption,True,+0.000000000,+0.000000000
degenerate_one_layer,True,+0.000000000,+0.000000000
interpolation_missing_values,True,+0.000000000,+0.000000000


Reproducibility check passed: frozen v0 results match baseline-v0.json.


## 7. Optimization ladder

Versions 1–18 implement these changes cumulatively. Version 19 deliberately branches from v15,
isolating its terminal-stage experiment from rejected layers 16–18; version 20 extends that branch
backward through D PROC.

| Step | Implemented change | Primary scenarios | Expected signal |
|---:|---|---|---|
| 0 | FIFO singleton baseline | all | legal reference point |
| 1 | Allow multiple unfinished requests per cloud | `two_cloud_parallel`, `output_length_skew` | less cloud idle time, lower elapsed time |
| 2 | Assign new requests using current cloud load | `output_length_skew` | less reservation/load imbalance |
| 3 | Group decode-ready work immediately | `batch_friendly_burst` | higher throughput and score |
| 4 | Select group sizes from the task-time table | `nonmonotonic_batch_table`, interpolation | avoid groups whose per-item service rate is worse |
| 5 | Add conservative SLO urgency and bounded waiting | `slo_priority_collision`, `latency_sensitive_stream` | protect aged requests without destroying throughput |
| 6 | Split long `P PROC` work into layer pieces | `single_cloud_prefill_interleave` | let ready decode work interleave between pieces |
| 7 | Add score- and link-aware ordering/group cost | `latency_weighted_slow_link` | improve TDR when latency dominates the score |
| 8 | Track exact virtual resource/link finish times | `exact_wait_horizon` | avoid event waits that overshoot their budget |
| 9 | Price D PRE cloud fanout and compatible cohorts | `cross_cloud_fanout` | reduce fixed UP latencies per group |
| 10 | Add batching affinity to permanent cloud placement | `batch_aware_placement` | choose pack versus spread from the decode curve |
| 11 | Predict TDR and next-token slack | `predicted_deadline_slack` | advance the most valuable overdue milestone |
| 12 | Tie prefill pieces to decode events/deadlines | `chunk_deadline_collision` | reduce decode blocking without tiny pieces |
| 13 | Use attained service and completed-output history | `attained_service_tail` | give likely-short/young streams a bounded opportunity |
| 14 | Penalize injection into congested links | `downstream_backpressure` | drain downstream work under queue pressure |
| 15 | Look through a hostile downstream decode curve | `one_token_lookahead` | avoid locally fast but end-to-end slow groups |
| 16 | Score bounded counterfactual group candidates | `counterfactual_grouping` | compare complete next-token paths with a v15 fallback |
| 17 | Fit the group-value coefficients offline | `learned_grouping_recovery` | retain only train-selected coefficients and audit holdout |
| 18 | Test nonlinear group-feature interactions | `nonlinear_ranker_holdout` | accept the selected zero-interaction null result |
| 19 | Simulate finite D POST queue clearance with known arrivals | `terminal_dpost_remainder`, `terminal_dpost_future_arrival` | enlarge only when modeled clearance and score both improve |
| 20 | Roll D PROC through FIFO DOWN and finite D POST clearance | `terminal_dproc_clearance` | change only in the one-cloud regime the rollout fully models |

The benchmark workbench measures both each version versus v0 and each layer versus its recorded
predecessor. That comparison is the cleanest local evidence for the effect of one feature gate.

### How the layers fit together

Layers 1–18 are cumulative rather than unrelated schedulers; layer 19 branches from v15 and layer
20 extends that branch. At every event frame the scheduler follows the same correctness loop—
consume all events, update state, identify free resources, and emit only legal work.

| Decision | Layers that affect it | Question being answered |
|---|---|---|
| Cloud admission | 1, 2, 7, 10 | Which cloud should own a request, and should admission be deferred? |
| Ready-task ordering | 5, 7, 8, 11, 14 | Which legal stage should use a free server next? |
| Decode group formation | 3, 4, 5, 7, 9, 13, 15, 16, 17, 18, 19, 20 | Which members and size should share the next decode task? |
| Prefill execution granularity | 6, 12 | Should one long `P PROC` run fully or expose deadline-aware interleaving points? |
| Short-horizon prediction | 8, 11, 14, 15, 16, 19, 20 | What known resource, FIFO-link, and downstream costs follow this action? |
| Offline policy calibration | 17, 18 | Which observable-feature weights survive train/holdout validation? |

None of these layers predicts the hidden output length. They use only information already
revealed by the protocol: arrival time, input length, cloud association, request state,
ready queues, resource busy state, the supplied task-time table, scoring weights/SLOs, and
known in-flight work.

### Layer 0 — frozen FIFO singleton baseline

**Limitation we start with.** The baseline reserves a cloud from one request's `P PRE` until
that request's final `FIN`. It also sends every decode stage as a singleton group and runs the
whole prefill-processing range `[0, num_layers)` in one task. This is intentionally stricter
than the problem.

**Intuition.** It is an excellent correctness reference because ownership is simple: one
request, one cloud, one lifecycle. But a reserved cloud can sit idle while its request is on
the edge or a shared transfer link. The remaining layers remove those artificial idle periods
while preserving the real rule that a server executes at most one task at a time.

**Why keep it forever?** Without a frozen v0, we could see that today's scheduler is “fast”
but could not reproduce what changed. Exact transcript tests remain attached to this version;
optimized schedules are checked for legality rather than identical command order.

### Layer 1 — multiple active requests per cloud

**Baseline bottleneck.** “One task running on a cloud” and “one unfinished request assigned to
a cloud” are different constraints. Only the first is real. The baseline incorrectly couples
them, so cloud compute goes idle whenever its one request moves through edge or transfer work.

**Policy.** A cloud may own many unfinished requests, each with its own state, while its compute
resource remains protected by one `cloud_busy` flag. New requests are assigned round-robin.
Cloud-ready `P PROC` and `D PROC` stages wait in per-cloud queues; the cloud still dispatches
only one assignment in a frame.

**Why it helps.** This creates a pipeline. While request A is uploading or doing edge work,
the same cloud can process request B. We increase the chance that every free cloud has legal
work ready, which usually raises throughput and lowers total elapsed time.

**Correctness invariants.** A request keeps the cloud chosen by `P PRE`; every `D PROC` group
remains single-cloud; and a busy cloud never receives a second concurrent task.

**Tradeoff.** Round-robin balances request counts, not work. Because output lengths are hidden,
one cloud can accumulate long-lived decode streams while another receives short requests.
Queueing can also worsen an individual request's TPOT even when total throughput improves.

### Layer 2 — observable-load-aware cloud placement

**Layer-1 bottleneck.** Two clouds with three requests each need not have equal work. Input
lengths are visible, ready queues differ, and one cloud may already be busy. Pure round-robin
ignores all of that.

**Policy.** Before `P PRE`, estimate each cloud's outstanding work and choose the minimum:

$$
\text{load}(c) = \text{remaining busy time}
+ \text{known prefill work}
+ \text{ready decode work}
+ 0.35\,\text{active-request proxy}.
$$

The proxy prices each unfinished request as a fraction of one singleton `D PROC` service cost.
It is deliberately modest because the true remaining output tokens are unknowable.

**Why it helps.** The first three terms route visible work away from a cloud that cannot serve
it soon. The active-request term prevents a cloud with little currently-ready work—but many
requests temporarily on the edge or links—from looking falsely empty.

**Tradeoff.** This is an estimate, not clairvoyance. A request with one hidden token and a
request with one thousand hidden tokens look the same before `FIN` evidence arrives. The
coefficient can therefore under- or over-price future decode load, and placement is permanent
because requests cannot migrate clouds later.

### Layer 3 — immediately group compatible decode work

**Singleton bottleneck.** Every task pays scheduling overhead `S`. If eight ready requests run
as eight singleton `D PROC` tasks, the cloud pays `S` eight times. A group pays it once and may
also receive a sublinear task duration from the supplied table.

**Policy.** When a decode resource becomes free, group all compatible requests that are ready
*right now*:

- `D PRE` and `D POST` may combine requests from different clouds because they run on edge `E`;
- `D PROC` combines only requests assigned to that particular cloud;
- the group lasts for one stage of one decode iteration—membership is recalculated later.

**Why it helps.** Grouping amortizes `S` and converts many queue operations into one service
interval. In a synchronized burst this can produce a large throughput gain and much smaller
token gaps.

**Why “immediate” matters.** This layer does not wait for a hypothetical future member. It
captures batching efficiency without yet risking deliberate idle time.

**Tradeoff.** “All ready requests” is not automatically the best size. A large group can occupy
the edge/cloud longer, delay urgent work, and perform poorly if the task-time table becomes
inefficient at larger batch sizes. That motivates layer 4.

### Layer 4 — task-table-aware group size

**Layer-3 bottleneck.** The largest ready group minimizes the number of assignments, but the
supplied duration curve can be nonmonotonic. If `T(8)` is much more than twice `T(4)`, two
groups of four can beat one group of eight despite paying `S` twice.

**Policy.** For each decode stage independently, choose the currently available size `b` that
maximizes local service rate:

$$
\text{rate}(b) = \frac{b}{S + T_{\text{stage}}(b)}.
$$

Candidates include size 1, all currently ready members, and task-table breakpoints plus their
immediate neighbors. Missing `-1` entries are ignored and usable values are interpolated.
Smaller groups win exact rate ties, limiting unnecessary convoy size.

**Why it helps.** The choice comes from the machine's supplied performance curve rather than
an assumption that batching always scales. The `nonmonotonic_batch_table` case intentionally
makes size 8 bad so this layer can demonstrate choosing efficient groups of 4.

**Tradeoff.** This optimizes one stage's local members-per-millisecond, not the complete request
network. It does not know future arrivals and may leave a remainder group. Queueing, link
contention, and request SLOs can make the globally best decision differ from the local rate.

### Layer 5 — SLO-aware urgency and tightly bounded waiting

This layer addresses two opposite mistakes: always following FIFO when a request is already
late, and always dispatching immediately when a nearly complete efficient group is about to
become ready.

**Urgency policy.** Normalize observed age by the relevant target:

$$
u_{\text{prefill}} = \frac{\text{now} - \text{arrival}}{\text{SLO1}},\qquad
u_{\text{decode}} = \frac{\text{now} - \text{decode-clock start}}{\text{SLO2}}.
$$

Under strongly latency-weighted scoring (`w_c > 0.8`), tasks with `u >= 1` may move ahead of
ordinary FIFO work. Otherwise ready sequence remains the primary order. This conservative gate
avoids turning every small age difference into priority churn.

**Controlled-wait policy.** Waiting is considered only when throughput weight is at least
`0.95`, a known future event will wake the scheduler, the desired table-aware group is larger
than the current group, the oldest member has consumed less than half its TPOT budget, and the
elapsed wait is inside a small SLO2-derived budget. `D POST` is never held for batching because
it is already the final step that reveals progress or `FIN`.

**Why it helps.** Urgency protects requests near/over an SLO boundary; bounded waiting can trade
a little idle time for enough additional members to amortize `S`. The supplied scoring weights
decide which behavior is even eligible.

**Tradeoff.** The protocol provides event-driven wakeups, not self-set timers. The next event
can occur later than the nominal wait budget, so waiting must remain narrow. Priority also
changes who waits rather than eliminating work; improving TPOT for one request can delay another.

### Layer 6 — adaptive, gap-free prefill chunks

**Long-task bottleneck.** A legal full `P PROC 0 num_layers` can occupy one cloud for a long
interval. Tasks cannot be preempted after dispatch, so decode work becoming ready one moment
later must wait for the entire prefill.

**Policy.** For models with more than eight layers, split `P PROC` into contiguous pieces only
when the same cloud has competing decode or prefill work. Piece size targets roughly:

$$
\max\left(4S,\ \min(0.25\,\text{SLO1},\ 0.5\,\text{SLO2})\right)
$$

milliseconds of processing, converted proportionally into a number of model layers. Every
piece starts exactly where the previous one ended; the final end is `num_layers`.

**Why it helps.** Chunk boundaries are scheduling opportunities. After one piece completes,
the cloud can run ready `D PROC` work before resuming prefill, reducing head-of-line blocking
without violating the no-preemption rule.

**Tradeoff.** Every piece pays `S`. Chunks that are too small destroy throughput; chunks that
are too large recreate the blocking problem. We therefore keep full prefills for small models
or when there is no competing work, and require a target of at least `4S`.

### Layer 7 — score- and shared-link-aware scheduling

**Compute-only bottleneck.** The clouds are separate compute resources, but all of them share
collective FIFO `UP` and `DOWN` links. A placement or group that looks efficient on compute can
inject a large transfer ahead of latency-sensitive traffic and dominate TDR/TPOT.

**Policy components.** This layer is intentionally conditional:

1. **Transfer-aware group cost.** For `D PRE` and `D PROC`, group-size service cost also includes
   an estimated transfer time, so compute batching does not appear free on a slow link.
2. **Latency-weighted prefill ordering.** When latency weight exceeds throughput weight, inspect
   a bounded FIFO window and prefer short prefill transfers. Request age subtracts from that
   cost, preventing large old requests from being ignored forever.
3. **Downstream stage preference under link pressure.** On constrained links, edge ordering
   favors `D POST → P POST → D PRE → P PRE`, while clouds favor ready `D PROC` over new `P PROC`.
   This tends to finish already-invested work before admitting another large transfer.
4. **Narrow admission pacing.** Under strongly latency-weighted scoring, a very young prefill
   may be deferred when the existing upload backlog already exceeds the TDR target and another
   known event will wake the scheduler.

When throughput weight dominates, admission preserves FIFO; shortest-transfer-first is not a
universal rule.

**Why it helps.** TDR-sensitive workloads benefit from completing small/advanced requests before
a huge upload monopolizes the FIFO link. Incorporating transfer cost also prevents selecting a
group solely because its compute curve looks fast.

**Tradeoff.** Shortest-transfer-first can postpone large requests, and prioritizing first-token
readiness can make inter-token gaps worse. The `latency_weighted_slow_link` result demonstrates
exactly that trade: a large TDR improvement raises score under its weights even though TPOT
worsens. This is why the layer is gated by scoring emphasis rather than always enabled equally.

### The overall intuition in one sentence

Keep every resource doing useful legal work, amortize fixed overhead when compatible work is
ready, size batches from measured curves, create safe interleaving points around long tasks,
and spend latency only when the scoring weights say the throughput benefit is worth it.

The policies are heuristics because future arrivals and output lengths are hidden. Their value
must be judged scenario by scenario, including regressions—not inferred from one aggregate mean.

In [16]:
registry = json.loads(REGISTRY_PATH.read_text())
display_table(
    [
        {
            "layer": version.get("layer", 0),
            "version": version["name"],
            "compile gate": ", ".join(version.get("compile_defines", [])) or "standalone",
            "description": version["description"],
        }
        for version in registry["versions"]
        if version["name"] != "working-tree"
    ]
)

layer,version,compile gate,description
0,v0-baseline,standalone,Frozen FIFO singleton baseline before scoring optimizations.
1,v1-multi-active,OPT_LEVEL=1,Multiple unfinished singleton requests per cloud with round-robin assignment.
2,v2-load-aware,OPT_LEVEL=2,"Adds cloud assignment using busy time, known prefill work, ready decode work, and an active-request proxy."
3,v3-immediate-groups,OPT_LEVEL=3,"Adds immediate grouping for ready D PRE, same-cloud D PROC, and D POST work."
4,v4-table-groups,OPT_LEVEL=4,Selects decode group size by interpolated task-table service rate instead of always taking every ready member.
5,v5-slo-aware,OPT_LEVEL=5,Adds conservative SLO urgency and tightly gated controlled waiting when future events are known.
6,v6-prefill-chunks,OPT_LEVEL=6,Adds adaptive gap-free P PROC layer chunks when longer prefills compete with other cloud work.
7,v7-link-aware,OPT_LEVEL=7,Adds transfer-aware group cost and latency-weighted shortest-prefill ordering while preserving FIFO on throughput-weighted links.
8,v8-exact-timelines,OPT_LEVEL=8,Tracks exact virtual resource and FIFO-link finish times and bounds batching waits by the next known event.
9,v9-fanout-cohorts,OPT_LEVEL=9,Adds cloud-fanout-aware D PRE group formation and waits only for compatible decode cohort events.


### Where the optimization decisions live

These bounded excerpts are the decision points—not copies of the whole scheduler. Rerunning
the notebook always reads the checked-in C++.

In [17]:
optimization_excerpts = [
    ("Load-aware cloud selection (layers 2/10)", "double cloud_load_score", "double observed_request_urgency"),
    ("Table-aware group size (layer 4)", "int best_group_size", "bool should_wait_for_group"),
    ("Request urgency (layer 5)", "double request_urgency", "int edge_stage_rank"),
    ("Bounded waiting (layer 5)", "bool should_wait_for_group", "bool should_defer_prefill_admission"),
    ("Adaptive prefill chunks (layer 6)", "int choose_prefill_piece_end", "vector<Candidate> cloud_candidates"),
    ("Latency-weighted prefill ordering (layer 7)", "int take_link_aware_prefill_request", "double cloud_load_score"),
    ("Exact virtual timelines (layer 8)", "void enqueue_transfer", "void complete_transfer"),
    ("Fanout-aware D PRE groups (layer 9)", "vector<int> choose_d_pre_members", "bool should_wait_for_group"),
    ("Predicted path slack (layer 11)", "double estimated_prefill_path", "double action_service_time"),
    ("Attained-service selection (layer 13)", "double expected_remaining_tokens", "double estimated_prefill_path"),
    ("Backpressure and lookahead (layers 14/15)", "double downstream_pressure", "double decode_member_value"),
    ("Counterfactual grouping (layer 16)", "vector<int> bounded_candidate_group_sizes", "vector<int> choose_d_pre_members"),
    ("Learned group value (layers 17/18)", "double counterfactual_group_value", "vector<int> choose_counterfactual_decode_group"),
    ("Finite D POST queue clearance (layer 19)", "vector<int> terminal_dpost_members", "bool should_wait_for_group"),
    ("D PROC-to-D POST clearance (layer 20)", "vector<int> terminal_dproc_members", "vector<int> legacy_d_pre_members"),
]
for title, start_marker, end_marker in optimization_excerpts:
    display(Markdown(f"#### {title}"))
    display(Code(source_between(layered_source, start_marker, end_marker, max_lines=90), language="cpp"))

#### Load-aware cloud selection (layers 2/10)

double cloud_load_score(int cloud) const {
        const double remaining_busy = max(0.0, cloud_busy_until_[cloud] - current_time_);
        const double decode_proxy =
            active_requests_[cloud] * (schedule_cost_ + duration(DurationColumn::DECODE_PROC, 1));
        const int ready_decode = static_cast<int>(d_proc_ready_[cloud].size());
        const double ready_decode_work =
            ready_decode > 0
                ? schedule_cost_ + duration(DurationColumn::DECODE_PROC, ready_decode)
                : 0.0;
        return remaining_busy + pending_prefill_work_[cloud] + ready_decode_work +
               0.35 * decode_proxy;
    }

    double batch_aware_cloud_score(const Request& req, int cloud) const {
        (void)req;
        const int prospective_cohort = max(1, active_requests_[cloud] + 1);
        const int candidate_size = min(
            prospective_cohort,
            best_group_size(DurationColumn::DECODE_PROC, prospective_cohort)
        );
        const double singleton = schedule_cost_ + duration(DurationColumn::DECODE_PROC, 1);
        const double grouped_per_request =
            (schedule_cost_ + duration(DurationColumn::DECODE_PROC, candidate_size)) /
            candidate_size;
        const double savings_per_iteration = max(0.0, singleton - grouped_per_request);
        const double efficiency_ratio = singleton / max(1e-9, grouped_per_request);
        const double cohort_strength = min(
            4,
            active_decode_requests_[cloud] + static_cast<int>(d_proc_ready_[cloud].size())
        );
        const int minimum_active = *min_element(active_requests_.begin(), active_requests_.end());
        const bool seed_decode_cohort = minimum_active == 0 &&
                                        active_requests_[cloud] == 1 &&
                                        active_decode_requests_[cloud] > 0;
        const double credit_scale = seed_decode_cohort ? 1.0 : 0.2;
        const double credit_cap = seed_decode_cohort ? 2.0 * singleton : 0.5 * singleton;
        const double batch_credit =
            efficiency_ratio >= 3.0 && active_requests_[cloud] <= minimum_active + 1
            ? min(
                  credit_cap,
                  credit_scale * throughput_weight_ * (1.0 + 0.5 * cohort_strength) *
                      savings_per_iteration
              )
            : 0.0;
        return cloud_load_score(cloud) - batch_credit;
    }

    int choose_cloud(const Request& req) {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel == 1) {
            const int cloud = next_round_robin_cloud_;
            next_round_robin_cloud_ = (next_round_robin_cloud_ + 1) % cloud_count_;
            return cloud;
        }
        // SUBMISSION_FEATURE_END pre20_only

        int best_cloud = 0;
        double best_load = kOptimizationLevel >= 10
            ? batch_aware_cloud_score(req, 0)
            : cloud_load_score(0);
        for (int cloud = 1; cloud < cloud_count_; ++cloud) {
            const double load = kOptimizationLevel >= 10
                ? batch_aware_cloud_score(req, cloud)
                : cloud_load_score(cloud);
            if (load + 1e-12 < best_load) {
                best_load = load;
                best_cloud = cloud;
            }
        }
        return best_cloud;
    }

#### Table-aware group size (layer 4)

int best_group_size(DurationColumn column, int available) const {
        if (available <= 1 || kOptimizationLevel < 3) {
            return 1;
        }
        // SUBMISSION_FEATURE_BEGIN experimental_grouping
        if constexpr (kExperimentalGrouping) {
            const vector<int>& cache = best_group_size_cache_[static_cast<int>(column)];
            return cache[min<int>(available, cache.size() - 1)];
        }
        // SUBMISSION_FEATURE_END experimental_grouping
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel == 3) {
            return available;
        }
        // SUBMISSION_FEATURE_END pre20_only

        set<int> candidates = {1, available};
        for (const auto& [size, ignored] : duration_curves_[static_cast<int>(column)]) {
            (void)ignored;
            for (int candidate : {size - 1, size, size + 1}) {
                if (1 <= candidate && candidate <= available) {
                    candidates.insert(candidate);
                }
            }
        }

        int best_size = 1;
        double best_rate = -1;
        for (int size : candidates) {
            double service_time = schedule_cost_ + duration(column, size);
            // SUBMISSION_FEATURE_BEGIN pre20_only
            if constexpr (kOptimizationLevel >= 7) {
            // SUBMISSION_FEATURE_END pre20_only
                if (column == DurationColumn::DECODE_PRE ||
                    column == DurationColumn::DECODE_PROC) {
                    service_time += transfer_time(
                        static_cast<long long>(size) * bytes_per_token_
                    );
                }
            // SUBMISSION_FEATURE_BEGIN pre20_only
            }
            // SUBMISSION_FEATURE_END pre20_only
            // SUBMISSION_FEATURE_BEGIN pre20_only
            if constexpr (kOptimizationLevel >= 15) {
            // SUBMISSION_FEATURE_END pre20_only
                if (column == DurationColumn::DECODE_PRE &&
                    downstream_group_is_hostile(size)) {
                    service_time += schedule_cost_ +
                                    duration(DurationColumn::DECODE_PROC, size) +
                                    transfer_time(
                                        static_cast<long long>(size) * bytes_per_token_
                                    ) +
                                    schedule_cost_ +
                                    duration(DurationColumn::DECODE_POST, size);
                } else if (column == DurationColumn::DECODE_PROC &&
                           downstream_group_is_hostile(size)) {
                    service_time += schedule_cost_ +
                                    duration(DurationColumn::DECODE_POST, size);
                }
            // SUBMISSION_FEATURE_BEGIN pre20_only
            }
            // SUBMISSION_FEATURE_END pre20_only
            const double rate = size / service_time;
            if (rate > best_rate + 1e-12 ||
                (abs(rate - best_rate) <= 1e-12 && size < best_size)) {
                best_rate = rate;
                best_size = size;
            }
        }
        return best_size;
    }

    // SUBMISSION_FEATURE_BEGIN experimental_grouping
    vector<int> bounded_candidate_group_sizes(
        DurationColumn column,
        int available
    ) const {
        set<int> sizes = {1, available};
        const int best = best_group_size(column, available);
        for (int candidate : {
                 best - 1,
                 best,
                 best + 1,
                 best / 2,
                 min(available, 2 * best),
                 available / 4,
                 available / 2,
                 3 * available / 4,
             }) {
            if (1 <= candidate && candidate <= available) {
// ... bounded notebook preview ...

#### Request urgency (layer 5)

...

double counterfactual_group_value(const GroupEvaluation& group) const {
        double rate_weight = 1.15;
        double efficiency_weight = 0.40;
        double waiting_weight = 1.00;
        double urgency_weight = 0.40;
        double completion_weight = 0.10;
        double fanout_penalty = 0.20;
        double excluded_penalty = 0.28;
        double dispersion_penalty = 0.15;
        if constexpr (kOptimizationLevel >= 17) {
            rate_weight = GROUP_RATE_WEIGHT;
            efficiency_weight = GROUP_EFFICIENCY_WEIGHT;
            waiting_weight = GROUP_LATENCY_WEIGHT;
            urgency_weight = GROUP_URGENCY_WEIGHT;
            completion_weight = GROUP_COMPLETION_WEIGHT;
            fanout_penalty = GROUP_FANOUT_PENALTY;
            excluded_penalty = GROUP_EXCLUDED_PENALTY;
            dispersion_penalty = GROUP_DISPERSION_PENALTY;
        }

        double value =
            throughput_weight_ *
                (rate_weight * group.normalized_rate +
                 efficiency_weight * group.service_efficiency) +
            latency_weight_ *
                (waiting_weight * group.waiting_quality +
                 urgency_weight * group.urgency_progress +
                 completion_weight * group.completion_potential) -
            fanout_penalty * group.fanout_pressure -
            excluded_penalty * group.excluded_pressure -
            dispersion_penalty * group.finish_dispersion;

        if constexpr (kOptimizationLevel >= 18) {
            const double uncongested = max(0.0, 1.0 - group.link_pressure);
            value += GROUP_INTERACTION_EFFICIENCY * throughput_weight_ *
                     group.service_efficiency * uncongested;
            value += GROUP_INTERACTION_URGENCY * latency_weight_ *
                     group.urgency_progress * (1.0 - group.waiting_quality);
            value -= GROUP_INTERACTION_CONGESTION *
                     (group.fanout_pressure * group.link_pressure +
                      group.excluded_pressure * max(0.0, group.urgency_progress - 1.0));
        }
        return value;
    }

#### Finite D POST queue clearance (layer 19)

vector<int> terminal_dpost_members() {
        vector<int> ready = collect_ready(d_post_ready_, RequestState::READY_D_POST);
        if (ready.empty()) {
            fail("terminal D POST selection found no ready members");
        }
        stable_sort(ready.begin(), ready.end(), [&](int left, int right) {
            const double left_value = decode_member_value(request(left), TaskKind::D_POST);
            const double right_value = decode_member_value(request(right), TaskKind::D_POST);
            if (abs(left_value - right_value) > 1e-12) {
                return left_value > right_value;
            }
            return request(left).ready_sequence < request(right).ready_sequence;
        });

        const int available = static_cast<int>(ready.size());
        const int fallback_size = best_group_size(
            DurationColumn::DECODE_POST, available
        );
        auto prefix = [&](int size) {
            return vector<int>(ready.begin(), ready.begin() + size);
        };
        if (available <= 1 || available > 96 || fallback_size <= 1 ||
            latency_weight_ <= 0.1 || distance_baseline_ <= 0) {
            return prefix(fallback_size);
        }

        set<int> sizes = {1, fallback_size, available};
        for (int size : {
                 fallback_size - 1,
                 fallback_size + 1,
                 fallback_size / 2,
                 min(available, 2 * fallback_size),
                 available / 4,
                 available / 2,
                 3 * available / 4,
             }) {
            if (1 <= size && size <= available) {
                sizes.insert(size);
            }
        }
        const vector<pair<int, double>>& curve =
            duration_curves_[static_cast<int>(DurationColumn::DECODE_POST)];
        for (int anchor : {fallback_size, available / 2}) {
            auto position = lower_bound(
                curve.begin(), curve.end(),
                make_pair(anchor, -numeric_limits<double>::infinity())
            );
            for (int offset = -2; offset <= 2; ++offset) {
                const long long index = distance(curve.begin(), position) + offset;
                if (0 <= index && index < static_cast<long long>(curve.size())) {
                    sizes.insert(min(available, curve[index].first));
                }
            }
        }

        vector<pair<double, int>> future_arrivals;
        int future_members = 0;
        for (const TransferPrediction& transfer : predicted_down_queue_) {
            if (!transfer.decode || transfer.finish_time < current_time_ - 1e-12 ||
                future_arrivals.size() >= 8 || future_members >= 96 - available) {
                continue;
            }
            const int members = min<int>(
                max<long long>(1, transfer.size_bytes / max<long long>(1, bytes_per_token_)),
                96 - available - future_members
            );
            if (members > 0) {
                future_arrivals.push_back({transfer.finish_time, members});
                future_members += members;
            }
        }

        auto value = [&](int first_size) {
            int completed_ready = 0;
            int queued = available;
            size_t arrival_index = 0;
            bool first_group = true;
            double virtual_time = current_time_;
            double gap_sum = observed_tpot_sum_;
            long long gap_count = observed_tpot_count_;
            while (queued > 0 || arrival_index < future_arrivals.size()) {
                if (queued == 0) {
                    virtual_time = max(
                        virtual_time, future_arrivals[arrival_index].first
                    );
                    while (arrival_index < future_arrivals.size() &&
                           future_arrivals[arrival_index].first <= virtual_time + 1e-12) {
                        queued += future_arrivals[arrival_index].second;
                        ++arrival_index;
                    }
// ... bounded no

#### D PROC-to-D POST clearance (layer 20)

vector<int> terminal_dproc_members(int cloud) {
        vector<int> ready = collect_ready(
            d_proc_ready_[cloud], RequestState::READY_D_PROC
        );
        if (ready.empty()) {
            fail("terminal D PROC selection found no ready members");
        }
        stable_sort(ready.begin(), ready.end(), [&](int left, int right) {
            const double left_value = decode_member_value(request(left), TaskKind::D_PROC);
            const double right_value = decode_member_value(request(right), TaskKind::D_PROC);
            if (abs(left_value - right_value) > 1e-12) {
                return left_value > right_value;
            }
            return request(left).ready_sequence < request(right).ready_sequence;
        });

        const int available = static_cast<int>(ready.size());
        const int fallback_size = best_group_size(
            DurationColumn::DECODE_PROC, available
        );
        auto prefix = [&](int size) {
            return vector<int>(ready.begin(), ready.begin() + size);
        };
        if (available <= 1 || available > 96 || cloud_count_ != 1 ||
            throughput_weight_ < 0.95 ||
            (latency_weight_ > 0 && distance_baseline_ <= 0)) {
            return prefix(fallback_size);
        }

        set<int> sizes = {fallback_size, available};
        for (int size : {
                 fallback_size - 1,
                 fallback_size + 1,
                 2 * fallback_size,
                 available / 4,
                 available / 2,
                 3 * available / 4,
             }) {
            if (fallback_size <= size && size <= available) {
                sizes.insert(size);
            }
        }
        const vector<pair<int, double>>& proc_curve =
            duration_curves_[static_cast<int>(DurationColumn::DECODE_PROC)];
        for (int anchor : {fallback_size, available / 2}) {
            auto position = lower_bound(
                proc_curve.begin(), proc_curve.end(),
                make_pair(anchor, -numeric_limits<double>::infinity())
            );
            for (int offset = -2; offset <= 2; ++offset) {
                const long long index = distance(proc_curve.begin(), position) + offset;
                if (0 <= index && index < static_cast<long long>(proc_curve.size())) {
                    const int candidate = min(available, proc_curve[index].first);
                    if (candidate >= fallback_size) {
                        sizes.insert(candidate);
                    }
                }
            }
        }

        vector<pair<double, int>> future_up;
        int future_up_members = 0;
        for (const TransferPrediction& transfer : predicted_up_queue_) {
            if (!transfer.decode || transfer.remote != cloud ||
                transfer.finish_time < current_time_ - 1e-12 ||
                future_up.size() >= 8 || future_up_members >= 96 - available) {
                continue;
            }
            const int members = min<int>(
                max<long long>(1, transfer.size_bytes / max<long long>(1, bytes_per_token_)),
                96 - available - future_up_members
            );
            if (members > 0) {
                future_up.push_back({transfer.finish_time, members});
                future_up_members += members;
            }
        }

        struct Arrival {
            double time;
            vector<int> items;
        };
        struct PendingDown {
            double proc_finish;
            vector<int> items;
        };
        auto value = [&](int first_size) {
            vector<Arrival> post_arrivals;
            if (!d_post_ready_.empty()) {
                post_arrivals.push_back(
// ... bounded notebook preview ...

### Optimization experiment worksheet

Copy this template into a new Markdown cell for each change:

```text
Optimization:
Hypothesis:
Code path changed:
Correctness invariant at risk:
Primary scenario:
Expected metric movement:
Observed result:
Interpretation:
Keep, revise, or revert:
```

## 8. Compare the current promoted v53 scheduler with v0

This compact comparison answers “did the accumulated policy help?” The companion benchmark
notebook performs the more diagnostic registered-lineage comparison through v20. Layers 16–18
remain rejected experiments; this section evaluates the layer-20 terminal branch plus the
promoted v25 resumed-prefill guard, v27 D POST threshold, v33 stage-correct cohort wait, and
v41's fresh-audited coherent decode cohort gate, v43's bounded P POST cohort seed, and v53's
sealed global coherent-DPOST gate. v53 groups the final D POST only when one D PRE group contains
every known unfinished request and public timing tables bound both transfer cost and predicted
member-ready dispersion. It never uses hidden output lengths.

In [18]:
def safe_label(label: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_-]+", "-", label).strip("-")


def run_policy_suite(label: str, executable: Path) -> list[dict[str, Any]]:
    executable = executable.resolve()
    if not executable.is_file():
        raise FileNotFoundError(executable)
    result_path = BUILD_DIR / f"notebook-{safe_label(label)}-results.json"
    run_checked(
        [
            "python3",
            "tools/local_judge.py",
            "--solver",
            str(executable),
            "--scenarios",
            str(SCENARIO_DIR),
            "--json-out",
            str(result_path),
        ]
    )
    return json.loads(result_path.read_text())


policy_results: dict[str, list[dict[str, Any]]] = {
    "baseline-v0": suite_results,
    "current-v53": run_policy_suite("current-v53", WORKING_SOLVER),
}

print("Loaded policies:", ", ".join(policy_results))

PASS official_worked_example      score= 500.000 tp=0.022222 tdr=30.000 tpot=0.000 elapsed=45.000
PASS single_sanity                score= 754.111 tp=0.081151 tdr=10.263 tpot=8.902 elapsed=36.968
PASS two_cloud_parallel           score=1000.000 tp=0.242413 tdr=28.048 tpot=13.597 elapsed=119.631
PASS output_length_skew           score= 708.206 tp=0.122735 tdr=23.249 tpot=11.294 elapsed=456.267
PASS batch_friendly_burst         score= 676.030 tp=0.227611 tdr=208.020 tpot=36.179 elapsed=562.363
PASS latency_sensitive_stream     score= 996.673 tp=0.279193 tdr=20.848 tpot=18.374 elapsed=214.905
PASS link_bottleneck              score= 241.004 tp=0.001395 tdr=12860.667 tpot=371.167 elapsed=22935.400
PASS prefill_preemption           score= 730.802 tp=0.097705 tdr=161.986 tpot=10.657 elapsed=655.035
PASS degenerate_one_layer         score=1000.000 tp=0.159849 tdr=9.159 tpot=7.967 elapsed=37.535
PASS interpolation_missing_values score= 879.952 tp=0.154382 tdr=26.086 tpot=10.144 elapsed=64.774


In [19]:
def comparison_against_baseline(
    baseline_rows: list[dict[str, Any]], candidate_rows: list[dict[str, Any]]
) -> list[dict[str, Any]]:
    baseline_map = {row["scenario"]: row for row in baseline_rows}
    candidate_map = {row["scenario"]: row for row in candidate_rows}
    rows = []
    for scenario_name, old in baseline_map.items():
        new = candidate_map[scenario_name]
        if not new.get("legal", False):
            rows.append({"scenario": scenario_name, "status": "ILLEGAL"})
            continue
        rows.append(
            {
                "scenario": scenario_name,
                "status": "legal",
                "score delta": f"{new['score'] - old['score']:+.3f}",
                "throughput %": f"{100 * (new['throughput'] / old['throughput'] - 1):+.1f}%",
                "TDR %": f"{100 * (new['tdr'] / old['tdr'] - 1):+.1f}%" if old["tdr"] else "n/a",
                "TPOT %": f"{100 * (new['tpot'] / old['tpot'] - 1):+.1f}%" if old["tpot"] else "n/a",
                "elapsed %": f"{100 * (new['elapsed'] / old['elapsed'] - 1):+.1f}%",
            }
        )
    return rows


for policy_name, results in policy_results.items():
    if policy_name == "baseline-v0":
        continue
    display(Markdown(f"### {policy_name} vs baseline-v0"))
    display_table(comparison_against_baseline(suite_results, results))

### current-v53 vs baseline-v0

scenario,status,score delta,throughput %,TDR %,TPOT %,elapsed %
official_worked_example,legal,+0.000,+0.0%,+0.0%,n/a,+0.0%
single_sanity,legal,+0.000,+0.0%,+0.0%,+0.0%,+0.0%
two_cloud_parallel,legal,+223.061,+90.5%,-68.8%,+24.2%,-47.5%
output_length_skew,legal,-15.931,-4.1%,-75.9%,+7.2%,+4.3%
batch_friendly_burst,legal,+475.597,+332.7%,-78.5%,-47.3%,-76.9%
latency_sensitive_stream,legal,+404.688,+76.2%,-80.5%,+25.0%,-43.2%
link_bottleneck,legal,-2.115,+17.8%,-12.3%,+63.4%,-15.1%
prefill_preemption,legal,+26.374,+0.1%,-55.4%,+19.7%,-0.1%
degenerate_one_layer,legal,+233.629,+89.0%,-64.7%,+0.8%,-47.1%
interpolation_missing_values,legal,+118.905,+41.4%,-38.3%,+13.9%,-29.3%


## Checks

These assertions verify the lab itself:

- every registered scenario was discovered;
- every baseline and current-policy interaction was legal;
- the official calibration case reproduces 45 ms elapsed and TDR 30 ms;
- the score formula reconstruction matched the judge; and
- frozen v0 results matched the saved `baseline-v0` snapshot.

In [20]:
assert len(scenario_paths) >= 29
assert all(row["legal"] for row in suite_results)
assert all(row["legal"] for row in policy_results["current-v53"])
assert set(saved_baseline).issubset(current_baseline)

official = current_baseline["official_worked_example"]
assert math.isclose(official["elapsed"], 45.0, abs_tol=1e-9)
assert math.isclose(official["tdr"], 30.0, abs_tol=1e-9)
assert math.isclose(official["tpot"], 0.0, abs_tol=1e-9)

run_checked(["make", "transcript-test"])
print("All notebook checks passed.")

python3 tests/run_transcript_tests.py build/v0-baseline
PASS official_example
PASS two_cloud_fifo
All notebook checks passed.


## Next steps

Open `scheduler_benchmark_workbench.ipynb` next. Its incremental table tells us which exact
layer moved which exact scenario. Use that evidence to tune one feature gate at a time:

1. inspect the target scenario and its score components;
2. review any scenario-level regression, even when the suite mean rose;
3. change one threshold or policy rule;
4. rerun both legality validation and the full version workbench; and
5. keep the change only when its tradeoff matches the supplied scoring weights.

The local scenarios are deterministic mechanism tests, not a model of the official hidden
workload distribution. A local win is evidence that a mechanism works under those inputs;
it is not a guarantee of leaderboard improvement.